# ASE practice — 3BPA

This notebook is a hands-on introduction to ASE using the real 3BPA extxyz dataset. The goal is to understand the `Atoms` object and its labels before touching MACE.

Work through the TODO cells in order. Use `EDA.ipynb` as a reference only after you have made a first attempt.

## 0. Setup

ASE is the Atomic Simulation Environment. Its central object is `Atoms`: a container for one molecular geometry, its element symbols, periodic-boundary information, and any attached calculator results.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from ase import Atoms
from ase.io import read

DATA_PATH = Path.cwd() / "data/dataset_3BPA/train_300K.xyz"
if not DATA_PATH.exists():
    DATA_PATH = Path.cwd().parent / "data/dataset_3BPA/train_300K.xyz"
if not DATA_PATH.exists():
    DATA_PATH = Path.cwd() / "papers/15-MACE/data/dataset_3BPA/train_300K.xyz"

print(DATA_PATH)

/Users/luvsuneja/Documents/repos/gdl-papers/papers/15-MACE/data/dataset_3BPA/train_300K.xyz


## 1. Read one frame

The extxyz file contains many frames one after another. Start with one frame.

TODO: use ASE's extxyz reader to load only the first frame. Then check whether the returned object is an `Atoms` instance and print its length.

Questions: What does `len(frame)` count? Why should it be 27?

In [4]:
# TODO(luv): read frame 0 from DATA_PATH with ASE.
frame = read(DATA_PATH, index=0, format='extxyz')

# TODO(luv): replace the ellipses with checks/prints that inspect the result.
assert isinstance(frame, Atoms)
assert len(frame) == 27
print(frame)

Atoms(symbols='C3HCON2H2CH2C2H2C3HCHCH3', pbc=False, cell=[50.0, 50.0, 50.0], calculator=SinglePointCalculator(...))


In [11]:
dir(frame)

['__add__',
 '__ase_optimizable__',
 '__class__',
 '__delattr__',
 '__delitem__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__iadd__',
 '__imul__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__mul__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_ase_handles_dynamic_stress',
 '_calc',
 '_celldisp',
 '_cellobj',
 '_centering_as_array',
 '_constraints',
 '_get_tokens_for_repr',
 '_masked_rotate',
 '_pbc',
 'append',
 'arrays',
 'ase_objtype',
 'calc',
 'cell',
 'center',
 'constraints',
 'copy',
 'edit',
 'euler_rotate',
 'extend',
 'fromdict',
 'get_all_distances',
 'get_angle',
 'get_angles',
 'get_angular_momentum',
 'get_array',
 'get_atomic_numbers',
 'get_calculator',
 'get_cell',
 'get_cell_lengths_and_angles',


## 2. Explore the `Atoms` object

An `Atoms` object is structured data, not just a matrix. Inspect each part and write down what it means in plain English.

TODO: inspect:

- the chemical symbols;
- the position array and its shape;
- the simulation cell and periodic-boundary flags;
- `frame.info.keys()`;
- `frame.arrays.keys()`.

Do not assume that every value in the header lives in `info`; ASE may attach calculated labels through a calculator.

In [13]:
frame.symbols

Symbols('C3HCON2H2CH2C2H2C3HCHCH3')

In [17]:

print(frame.positions.shape)
print(frame.positions)

(27, 3)
[[ 0.3265699  -1.01286015  0.72107275]
 [-0.19461567  0.25830309  0.8341389 ]
 [ 0.7924943  -1.60244652  1.90438013]
 [ 0.43514812 -1.59784402 -0.17897327]
 [-0.0698411   0.97877174  2.04804512]
 [-0.82551374  0.94764149 -0.23208227]
 [-0.2900658   2.37382353  1.86234713]
 [ 0.38837699  0.47664821  3.17063941]
 [-0.98416331  2.63164516  1.17783211]
 [-0.35613357  2.91935548  2.65951281]
 [ 0.72925588 -0.88036074  3.09652633]
 [ 0.89111308 -1.3321255   4.04892841]
 [ 1.2124948  -2.61699312  1.95761271]
 [-1.55059042  0.1399201  -1.1989431 ]
 [-2.9314348  -0.31868852 -0.84882155]
 [-0.88027102 -0.72270141 -1.50495635]
 [-1.71075243  0.76854244 -2.05655313]
 [-3.36709795 -1.6386861  -0.72393961]
 [-3.8819392   0.73797993 -0.59066374]
 [-4.66143774 -1.92714685 -0.2708153 ]
 [-2.71536973 -2.46157596 -0.92494411]
 [-5.57704224 -0.8782771  -0.11327951]
 [-4.95294558 -2.96156288 -0.08227495]
 [-5.17381471  0.43421272 -0.2990887 ]
 [-6.59525001 -1.1021947   0.20624618]
 [-5.9275935   1.

In [18]:
frame.cell

Cell([50.0, 50.0, 50.0])

In [19]:
frame.pbc

array([False, False, False])

In [12]:
# TODO(luv): print the five items listed above.
# Useful questions while inspecting: what is the shape of positions?
# Which fields are per-atom, and which describe the whole frame?
frame.info.keys()

dict_keys(['dihedrals'])

In [30]:
frame.info['dihedrals']

array([209.44,  80.54,  59.96])

In [21]:
frame.arrays.keys()

dict_keys(['numbers', 'positions'])

In [23]:
frame.numbers

array([6, 6, 6, 1, 6, 8, 7, 7, 1, 1, 6, 1, 1, 6, 6, 1, 1, 6, 6, 6, 1, 6,
       1, 6, 1, 1, 1])

## 3. Find the energy and forces

This dataset contains a scalar energy for the whole geometry and one force vector per atom. In ASE, extxyz labels such as these can be exposed through the attached single-point calculator.

TODO: retrieve the potential energy and forces using ASE's public accessors. Check that the energy is a scalar and that the force array has shape `(27, 3)`. Print the first three force vectors.

Remember the physical connection: $\mathbf F_a=-\nabla_{\mathbf r_a}E$.

In [24]:
# TODO(luv): retrieve energy and forces from frame.
energy = frame.get_potential_energy()
forces = frame.get_forces()

# TODO(luv): write shape/type checks before printing a small slice.
print(energy)
print(forces[:3])

-17660.461737861464
[[ 0.17092934  0.132742    0.79555789]
 [-0.89494012 -0.4486952  -1.90941638]
 [ 0.31093463  0.07569877  1.6261265 ]]


## 4. Read the complete trajectory

Now load every frame. This is not a train/test split: the file already defines the training collection.

TODO: read all frames and answer:

- How many frames are there?
- Does every frame have 27 atoms?
- Do the chemical symbols stay the same?
- What is the formula of the molecule?

Expected result: 500 frames, each with 27 atoms.

In [ ]:
# TODO(luv): use ASE's multi-frame index convention to load every frame.
all_frames = read(DATA_PATH, index=":")

# TODO(luv): turn the questions above into assertions and prints.
assert len(all_frames) == 500
assert all(len(atoms) == 27 for atoms in all_frames)

In [31]:
print(len(all_frames))

500


## 5. Geometry and label statistics

A frame changes its positions, energy, and forces. The number and identity of atoms stay fixed.

TODO: collect the 500 energies into a one-dimensional NumPy array. Collect the force magnitudes into an array with one row per frame and one column per atom. Report min/max energy and the largest force magnitude in the dataset.

Shape contract: `energies.shape == (500,)`; `force_norms.shape == (500, 27)`.

In [ ]:
frame.get_fo

In [42]:
# TODO(luv): build energies and force_norms from all_frames.
energies = np.array([f.get_potential_energy() for f in all_frames])
force_norms = np.array([np.mean(np.square(f.get_forces()), axis=1) for f in all_frames])
print(force_norms.shape)

assert energies.shape == (500,)
assert force_norms.shape == (500, 27)
print(energies.min(), energies.max())
print(force_norms.max())
print(force_norms.min())

(500, 27)
-17660.813926658517 -17659.76607569252
14.084975403378579
0.0001579978383398


## 6. Reproducible random sampling

The visualizer samples a frame so we can inspect different geometries. A seed makes the choice reproducible.

TODO: use NumPy's modern random generator to choose one valid frame index. Run the selection twice with the same seed and once with a different seed. What should match, and what should change?

In [45]:
# TODO(luv): implement seeded frame selection.
sampled_index_a = np.random.default_rng(seed=12).integers(0, 500)
sampled_index_b = np.random.default_rng(seed=12).integers(0, 500)

assert sampled_index_a == sampled_index_b
assert 0 <= sampled_index_a < len(all_frames)
print(sampled_index_a)

306


## 7. Make one frame tabular

Pandas is useful for answering questions atom-by-atom. Build a table with one row per atom and these columns:

`atom_index`, `symbol`, `x`, `y`, `z`, `fx`, `fy`, `fz`, `force_norm`.

TODO: create the table for the sampled frame. Then sort by `force_norm` and identify which atom feels the largest instantaneous force. Expected shape: `(27, 9)`.

In [57]:
force_norms.shape

(500, 27)

In [58]:
# TODO(luv): construct the requested pandas DataFrame from one ASE Atoms object.
atom_table = pd.DataFrame({
    "atom_index": list(range(27)),
    "symbol": frame.get_chemical_symbols(),
    "x": frame.arrays['positions'][:,0],
    "y": frame.arrays['positions'][:,1],
    "z": frame.arrays['positions'][:,2],
    "fx": frame.get_forces()[:,0],
    "fy": frame.get_forces()[:,1],
    "fz": frame.get_forces()[:,2],
    "force_norm": np.mean(np.square(frame.get_forces()), axis=1),
})

assert atom_table.shape == (27, 9)
print(atom_table.sort_values("force_norm", ascending=False).head())

    atom_index symbol         x         y         z        fx        fy  \
18          18      C -3.881939  0.737980 -0.590664  2.666044 -0.915446   
4            4      C -0.069841  0.978772  2.048045 -0.052357  2.218293   
9            9      H -0.356134  2.919355  2.659513 -0.043802  0.973301   
14          14      C -2.931435 -0.318689 -0.848822 -1.563550  1.023038   
10          10      C  0.729256 -0.880361  3.096526 -0.650109  1.941089   

          fz  force_norm  
18 -0.392239    2.699894  
4  -1.286384    2.192783  
9   2.120215    1.814849  
14  1.358061    1.778542  
10 -1.044927    1.760781  


## 8. Red-green helper exercise

Write a small helper that converts one ASE `Atoms` frame into the table from Section 7. Keep the helper focused: it should not read files or plot anything.

Contract for `frame_to_dataframe(atoms)`:

- input: one ASE `Atoms` object;
- output: a pandas DataFrame with 27 rows and the nine named columns;
- `force_norm` is the Euclidean norm of each `(fx, fy, fz)` row.

Implement the helper in the next cell. The following check is intentionally red until your implementation is complete.

In [60]:
def frame_to_dataframe(atoms):
    atom_table = pd.DataFrame({
    "atom_index": list(range(27)),
    "symbol": atoms.get_chemical_symbols(),
    "x": atoms.arrays['positions'][:,0],
    "y": atoms.arrays['positions'][:,1],
    "z": atoms.arrays['positions'][:,2],
    "fx": atoms.get_forces()[:,0],
    "fy": atoms.get_forces()[:,1],
    "fz": atoms.get_forces()[:,2],
    "force_norm": np.mean(np.square(atoms.get_forces()), axis=1),
})

    assert atom_table.shape == (27, 9)
    return atom_table    


In [61]:
# RED/GREEN CHECK — run after implementing frame_to_dataframe.
practice_table = frame_to_dataframe(all_frames[0])
assert practice_table.shape == (27, 9)
assert list(practice_table.columns) == [
    "atom_index", "symbol", "x", "y", "z",
    "fx", "fy", "fz", "force_norm",
]
assert np.allclose(
    practice_table[["fx", "fy", "fz"]].to_numpy(),
    all_frames[0].get_forces(),
)
print("helper checks passed")

helper checks passed


## 9. Capstone: connect back to the visualizer

Return to `EDA.ipynb` and compare its viewer with your understanding now. Without copying the whole function, make one small improvement:

- add the maximum per-atom force magnitude to the Plotly title; or
- add the atom's element and force magnitude to a pandas-backed report; or
- change the seed and explain why the geometry, energy, and force arrows change together.

The key data flow should now be clear: `extxyz -> ASE Atoms -> positions/energy/forces -> table or visualization`.